# 08 — Neural Network Foundations

In the previous notebook, we learned how to train a linear regression model from scratch using:

- Predictions
- Loss
- Gradients
- Gradient descent
- Autograd
- `nn.Linear`
- Optimizers

Now we will move from a single linear model to the foundation of modern deep learning:

> **Neural Networks**

A neural network is built from many simple computational units called **neurons**.

The power of neural networks comes from combining:

- Linear transformations
- Nonlinear activation functions
- Multiple layers
- Learnable parameters

## In this notebook, we will learn:

1. What is a neuron?
2. Inputs, weights, and bias
3. Linear transformation
4. Why linear models are limited
5. Activation functions
6. ReLU
7. Sigmoid
8. Tanh
9. Hidden layers
10. Output layers
11. Building a tiny neural network with raw tensors
12. Building the same network with `nn.Module`
13. Forward propagation
14. Parameter counting
15. Shape reasoning through layers
16. Common beginner mistakes
17. Practice exercises

## Main Goal

By the end of this notebook, you should understand the structure:

$$
\text{Input}
\rightarrow
\text{Linear Layer}
\rightarrow
\text{Activation}
\rightarrow
\text{Hidden Representation}
\rightarrow
\text{Output Layer}
$$

The most important question to ask at every layer is:

> **What is the input shape, what transformation happens, and what is the output shape?**


In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

print("PyTorch version:", torch.__version__)


# 1. What Is a Neuron?

A neuron is one of the basic computational units in a neural network.

A neuron receives:

- Input values
- Weights
- A bias

It computes a weighted sum:

$$
z=w_1x_1+w_2x_2+\cdots+w_nx_n+b
$$

Then an activation function may be applied:

$$
a=f(z)
$$

So a neuron has two main stages:

$$
\boxed{
\text{Linear Combination}
\rightarrow
\text{Activation}
}
$$


# 2. Inputs, Weights, and Bias

Suppose a neuron receives three inputs:

$$
x=
\begin{array}{|c|c|c|}
\hline
x_1 & x_2 & x_3 \\
\hline
\end{array}
$$

The neuron has three corresponding weights:

$$
w=
\begin{array}{|c|c|c|}
\hline
w_1 & w_2 & w_3 \\
\hline
\end{array}
$$

and one bias:

$$
b
$$

The pre-activation value is:

$$
z=x_1w_1+x_2w_2+x_3w_3+b
$$


## Example

Suppose:

$$
x=
\begin{array}{|c|c|c|}
\hline
1 & 2 & 3 \\
\hline
\end{array}
$$

$$
w=
\begin{array}{|c|c|c|}
\hline
0.5 & -1 & 2 \\
\hline
\end{array}
$$

and:

$$
b=0.1
$$

Then:

$$
z=(1)(0.5)+(2)(-1)+(3)(2)+0.1
$$

$$
=0.5-2+6+0.1
$$

$$
=\boxed{4.6}
$$


In [ ]:
x = torch.tensor([1.0, 2.0, 3.0])
w = torch.tensor([0.5, -1.0, 2.0])
b = torch.tensor(0.1)

z = torch.dot(x, w) + b

print("Neuron pre-activation:", z)


# 3. Vector Form of a Neuron

Instead of writing:

$$
w_1x_1+w_2x_2+\cdots+w_nx_n+b
$$

we can write:

$$
\boxed{z=x\cdot w+b}
$$

or with matrix notation:

$$
\boxed{z=xw^T+b}
$$

This compact notation is extremely important because neural networks perform many of these calculations at once.


# 4. From One Neuron to Multiple Neurons

Suppose we have one input vector with 3 features:

$$
x.shape=(3)
$$

and we want 2 neurons.

Each neuron needs its own weight vector of length 3.

So we can store the weights as:

$$
W.shape=(2,\ 3)
$$

where:

- 2 = number of neurons
- 3 = input features

The bias has shape:

$$
b.shape=(2)
$$

Then:

$$
z=xW^T+b
$$

Output shape:

$$
(3) @ (3,\ 2) + (2)
\rightarrow
\boxed{(2)}
$$


In [ ]:
x = torch.tensor([1.0, 2.0, 3.0])

W = torch.tensor([
    [0.5, -1.0, 2.0],
    [1.0,  0.5, -0.5]
])

b = torch.tensor([0.1, -0.2])

z = x @ W.T + b

print("Output:", z)
print("Output shape:", z.shape)


# 5. Linear Transformation

A layer with multiple neurons performs a **linear transformation**.

For a batch:

$$
X.shape=(batch,\ in\_features)
$$

Weights:

$$
W.shape=(out\_features,\ in\_features)
$$

Bias:

$$
b.shape=(out\_features)
$$

The transformation is:

$$
\boxed{Z=XW^T+b}
$$

Output shape:

$$
(batch,\ in)
@
(in,\ out)
+
(out)
$$

which gives:

$$
\boxed{(batch,\ out)}
$$


In [ ]:
X = torch.randn(5, 3)
W = torch.randn(4, 3)
b = torch.randn(4)

Z = X @ W.T + b

print("X shape:", X.shape)
print("W shape:", W.shape)
print("b shape:", b.shape)
print("Z shape:", Z.shape)


# 6. Why Linear Models Are Limited

Suppose we stack two linear transformations:

$$
h=W_1x+b_1
$$

and then:

$$
y=W_2h+b_2
$$

Substitute the first into the second:

$$
y=W_2(W_1x+b_1)+b_2
$$

Expand:

$$
y=(W_2W_1)x+(W_2b_1+b_2)
$$

This is still just another linear transformation.

So:

> **Stacking linear layers without nonlinear activations does not give us a truly more powerful nonlinear model.**

This is why activation functions are essential.


# 7. Why Nonlinearity Matters

Real-world relationships are often nonlinear.

Examples:

- Images
- Speech
- Medical signals
- Complex classification boundaries
- Language

A neural network becomes much more expressive when we apply a nonlinear activation after a linear layer.

The common pattern is:

$$
\boxed{
Z=XW^T+b
}
$$

followed by:

$$
\boxed{
A=f(Z)
}
$$

where:

$$
f
$$

is an activation function.


# 8. What Is an Activation Function?

An activation function transforms a neuron's pre-activation value.

Without activation:

$$
z=wx+b
$$

With activation:

$$
a=f(z)
$$

Common activation functions include:

- ReLU
- Sigmoid
- Tanh

Different activations are useful in different situations.


# 9. ReLU

ReLU stands for:

> **Rectified Linear Unit**

It is defined as:

$$
\boxed{
ReLU(x)=\max(0,x)
}
$$

That means:

- Negative values become `0`
- Positive values remain unchanged

Examples:

$$
\begin{array}{|c|c|}
\hline
x & ReLU(x) \\
\hline
-3 & 0 \\
\hline
-1 & 0 \\
\hline
0 & 0 \\
\hline
2 & 2 \\
\hline
5 & 5 \\
\hline
\end{array}
$$


In [ ]:
x = torch.tensor([-3.0, -1.0, 0.0, 2.0, 5.0])

relu = torch.relu(x)

print("Input:", x)
print("ReLU:", relu)


# 10. ReLU With `nn.ReLU`

PyTorch also provides:

`nn.ReLU()`


In [ ]:
relu_layer = nn.ReLU()

x = torch.tensor([-3.0, -1.0, 0.0, 2.0, 5.0])

print(relu_layer(x))


# 11. Visualizing ReLU

ReLU looks like:

$$
ReLU(x)=
\begin{cases}
0, & x<0 \\
x, & x\ge0
\end{cases}
$$


In [ ]:
x_values = torch.linspace(-5, 5, 200)
y_values = torch.relu(x_values)

plt.figure(figsize=(8, 5))
plt.plot(x_values.numpy(), y_values.numpy())
plt.axhline(0)
plt.axvline(0)
plt.xlabel("x")
plt.ylabel("ReLU(x)")
plt.title("ReLU Activation")
plt.show()


# 12. Why ReLU Is Popular

ReLU is widely used in hidden layers because it is:

- Simple
- Fast
- Nonlinear
- Effective in deep networks

A very common pattern is:

`Linear → ReLU → Linear → ReLU`

ReLU allows the network to learn nonlinear relationships.


# 13. Sigmoid

The sigmoid function is:

$$
\boxed{
\sigma(x)=\frac{1}{1+e^{-x}}
}
$$

Its output is always between:

$$
0
$$

and:

$$
1
$$

So:

$$
0<\sigma(x)<1
$$


In [ ]:
x = torch.tensor([-5.0, -2.0, 0.0, 2.0, 5.0])

sigmoid = torch.sigmoid(x)

print("Input:", x)
print("Sigmoid:", sigmoid)


# 14. Sigmoid Intuition

For large negative values:

$$
\sigma(x)\approx0
$$

At:

$$
x=0
$$

we get:

$$
\sigma(0)=0.5
$$

For large positive values:

$$
\sigma(x)\approx1
$$

Sigmoid is often used when we want to interpret an output as a probability for **binary classification**.

Later we will learn why training usually uses:

`BCEWithLogitsLoss`

instead of applying sigmoid manually before the loss.


In [ ]:
x_values = torch.linspace(-8, 8, 200)
y_values = torch.sigmoid(x_values)

plt.figure(figsize=(8, 5))
plt.plot(x_values.numpy(), y_values.numpy())
plt.axhline(0.5)
plt.axvline(0)
plt.xlabel("x")
plt.ylabel("Sigmoid(x)")
plt.title("Sigmoid Activation")
plt.show()


# 15. Tanh

Tanh stands for:

> **Hyperbolic Tangent**

Its output lies between:

$$
-1
$$

and:

$$
1
$$

So:

$$
-1<\tanh(x)<1
$$

The function is centered around zero.


In [ ]:
x = torch.tensor([-5.0, -2.0, 0.0, 2.0, 5.0])

tanh_output = torch.tanh(x)

print("Input:", x)
print("Tanh:", tanh_output)


In [ ]:
x_values = torch.linspace(-5, 5, 200)
y_values = torch.tanh(x_values)

plt.figure(figsize=(8, 5))
plt.plot(x_values.numpy(), y_values.numpy())
plt.axhline(0)
plt.axvline(0)
plt.xlabel("x")
plt.ylabel("tanh(x)")
plt.title("Tanh Activation")
plt.show()


# 16. Comparing ReLU, Sigmoid, and Tanh

$$
\begin{array}{|c|c|c|}
\hline
\textbf{Activation} & \textbf{Output Range} & \textbf{Common Use} \\
\hline
ReLU & [0,\infty) & \text{Hidden layers} \\
\hline
Sigmoid & (0,1) & \text{Binary probability outputs} \\
\hline
Tanh & (-1,1) & \text{Some hidden/recurrent settings} \\
\hline
\end{array}
$$

For beginner neural networks:

> **ReLU is a strong default choice for hidden layers.**


# 17. Hidden Layers

A **hidden layer** is a layer between the input and output.

For example:

$$
\text{Input}
\rightarrow
\text{Hidden Layer}
\rightarrow
\text{Output}
$$

Suppose:

$$
input\_features=3
$$

and the hidden layer has:

$$
4
$$

neurons.

Then:

$$
W_1.shape=(4,\ 3)
$$

and:

$$
b_1.shape=(4)
$$

The hidden representation is:

$$
h=ReLU(xW_1^T+b_1)
$$


In [ ]:
x = torch.randn(3)

W1 = torch.randn(4, 3)
b1 = torch.randn(4)

z1 = x @ W1.T + b1
h = torch.relu(z1)

print("Input shape:", x.shape)
print("Pre-activation shape:", z1.shape)
print("Hidden shape:", h.shape)


# 18. Output Layers

The **output layer** produces the final model output.

The number of output neurons depends on the task.

Examples:

$$
\begin{array}{|c|c|}
\hline
\textbf{Task} & \textbf{Typical Output Units} \\
\hline
\text{Regression} & 1 \\
\hline
\text{Binary classification} & 1 \\
\hline
\text{10-class classification} & 10 \\
\hline
\end{array}
$$

The output activation depends on the task and loss function.


# 19. A Tiny Neural Network

Let's build:

$$
3
\rightarrow
4
\rightarrow
2
$$

This means:

- 3 input features
- 4 hidden neurons
- 2 output neurons

Architecture:

$$
\boxed{
3
\rightarrow
4
\rightarrow
2
}
$$


# 20. Shape Reasoning for the Tiny Network

Input:

$$
x.shape=(3)
$$

First-layer weights:

$$
W_1.shape=(4,\ 3)
$$

First-layer bias:

$$
b_1.shape=(4)
$$

Hidden output:

$$
(3) @ (3,\ 4)+(4)
\rightarrow
\boxed{(4)}
$$

Second-layer weights:

$$
W_2.shape=(2,\ 4)
$$

Second-layer bias:

$$
b_2.shape=(2)
$$

Final output:

$$
(4) @ (4,\ 2)+(2)
\rightarrow
\boxed{(2)}
$$


# 21. Building the Network With Raw Tensors

We will implement:

$$
z_1=xW_1^T+b_1
$$

$$
h=ReLU(z_1)
$$

$$
z_2=hW_2^T+b_2
$$


In [ ]:
torch.manual_seed(42)

x = torch.randn(3)

W1 = torch.randn(4, 3)
b1 = torch.randn(4)

W2 = torch.randn(2, 4)
b2 = torch.randn(2)

z1 = x @ W1.T + b1
h = torch.relu(z1)
output = h @ W2.T + b2

print("x shape:", x.shape)
print("z1 shape:", z1.shape)
print("h shape:", h.shape)
print("output shape:", output.shape)

print("\\nOutput:")
print(output)


# 22. Forward Propagation

The process of moving data from input to output is called:

> **Forward propagation**

For our network:

$$
x
\rightarrow
z_1
\rightarrow
ReLU
\rightarrow
h
\rightarrow
z_2
$$

More explicitly:

$$
x
\rightarrow
xW_1^T+b_1
\rightarrow
ReLU
\rightarrow
hW_2^T+b_2
$$

The result is the network's prediction or raw output.


# 23. Batch Forward Propagation

Neural networks usually process multiple samples at once.

Suppose:

$$
X.shape=(5,\ 3)
$$

That means:

- 5 samples
- 3 features per sample

First layer:

$$
(5,\ 3) @ (3,\ 4)
\rightarrow
(5,\ 4)
$$

Second layer:

$$
(5,\ 4) @ (4,\ 2)
\rightarrow
(5,\ 2)
$$


In [ ]:
X = torch.randn(5, 3)

z1 = X @ W1.T + b1
h = torch.relu(z1)
output = h @ W2.T + b2

print("X shape:", X.shape)
print("Hidden shape:", h.shape)
print("Output shape:", output.shape)


# 24. Building the Same Network With `nn.Module`

PyTorch models are usually created by subclassing:

`nn.Module`

Let's build the same:

$$
3\rightarrow4\rightarrow2
$$

network.


In [ ]:
class TinyNetwork(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(3, 4)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(4, 2)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = TinyNetwork()

print(model)


# 25. Understanding `__init__`

Inside:

`__init__`

we define the layers that contain parameters.

For example:

`nn.Linear(3,4)`

contains:

$$
weight.shape=(4,\ 3)
$$

and:

$$
bias.shape=(4)
$$

The second layer:

`nn.Linear(4,2)`

contains:

$$
weight.shape=(2,\ 4)
$$

and:

$$
bias.shape=(2)
$$


In [ ]:
print("fc1 weight:", model.fc1.weight.shape)
print("fc1 bias:", model.fc1.bias.shape)

print("fc2 weight:", model.fc2.weight.shape)
print("fc2 bias:", model.fc2.bias.shape)


# 26. Understanding `forward()`

The `forward()` method defines how data moves through the network.

Our forward pass is:

```python
x = self.fc1(x)
x = self.relu(x)
x = self.fc2(x)
```

Conceptually:

$$
\text{Input}
\rightarrow
\text{Linear}
\rightarrow
\text{ReLU}
\rightarrow
\text{Linear}
\rightarrow
\text{Output}
$$


# 27. Calling the Model

In PyTorch, we usually call:

`model(x)`

rather than directly calling:

`model.forward(x)`

`model(x)` uses PyTorch's `nn.Module` machinery correctly.


In [ ]:
x = torch.randn(3)

output = model(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)
print("Output:", output)


# 28. Batch Input With `nn.Module`

The same model can process a batch.

Input shape:

$$
(32,\ 3)
$$

Output shape:

$$
(32,\ 2)
$$


In [ ]:
X = torch.randn(32, 3)

output = model(X)

print("Input shape:", X.shape)
print("Output shape:", output.shape)


# 29. Inspecting Model Parameters

We can inspect all model parameters using:

`model.named_parameters()`


In [ ]:
for name, parameter in model.named_parameters():
    print(name)
    print("shape:", parameter.shape)
    print("requires_grad:", parameter.requires_grad)
    print()


# 30. Parameter Counting

The number of parameters in a linear layer is:

$$
\boxed{
out\_features\times in\_features + out\_features
}
$$

The first term is for weights.

The second term is for biases.


## First Layer

For:

`nn.Linear(3,4)`

Weights:

$$
4\times3=12
$$

Biases:

$$
4
$$

Total:

$$
12+4=\boxed{16}
$$


## Second Layer

For:

`nn.Linear(4,2)`

Weights:

$$
2\times4=8
$$

Biases:

$$
2
$$

Total:

$$
8+2=\boxed{10}
$$

Complete model:

$$
16+10=\boxed{26}
$$

parameters.


In [ ]:
total_params = sum(p.numel() for p in model.parameters())

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)


# 31. Parameter Table

$$
\begin{array}{|c|c|c|c|}
\hline
\textbf{Layer} & \textbf{Weight Shape} & \textbf{Bias Shape} & \textbf{Parameters} \\
\hline
fc1 & (4,3) & (4) & 16 \\
\hline
fc2 & (2,4) & (2) & 10 \\
\hline
\textbf{Total} & - & - & \textbf{26} \\
\hline
\end{array}
$$


# 32. Shape Reasoning Through Layers

Suppose:

$$
X.shape=(32,\ 3)
$$

and the model is:

$$
3\rightarrow4\rightarrow2
$$

Then:

$$
\begin{array}{|c|c|}
\hline
\textbf{Stage} & \textbf{Shape} \\
\hline
Input & (32,3) \\
\hline
fc1 & (32,4) \\
\hline
ReLU & (32,4) \\
\hline
fc2 & (32,2) \\
\hline
Output & (32,2) \\
\hline
\end{array}
$$

Notice:

> ReLU changes values, but it does not change the tensor shape.


In [ ]:
X = torch.randn(32, 3)

a = model.fc1(X)
b = model.relu(a)
c = model.fc2(b)

print("Input:", X.shape)
print("After fc1:", a.shape)
print("After ReLU:", b.shape)
print("After fc2:", c.shape)


# 33. Why Hidden Size Matters

The number of neurons in a hidden layer is called the:

> **Hidden size**

For example:

$$
3\rightarrow4\rightarrow2
$$

uses hidden size `4`.

A larger hidden layer can represent more complex patterns, but it also adds more parameters and computation.

For example:

$$
3\rightarrow100\rightarrow2
$$

contains many more parameters than:

$$
3\rightarrow4\rightarrow2
$$

Bigger is not automatically better.


# 34. A Deeper Network

We can add more hidden layers.

Example:

$$
3\rightarrow8\rightarrow4\rightarrow2
$$


In [ ]:
class DeeperNetwork(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(3, 8)
        self.fc2 = nn.Linear(8, 4)
        self.fc3 = nn.Linear(4, 2)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

deep_model = DeeperNetwork()

print(deep_model)


In [ ]:
X = torch.randn(16, 3)

output = deep_model(X)

print("Input shape:", X.shape)
print("Output shape:", output.shape)


# 35. Shape Reasoning for the Deeper Network

For:

$$
X.shape=(16,\ 3)
$$

the transformations are:

$$
(16,\ 3)
\rightarrow
(16,\ 8)
\rightarrow
(16,\ 4)
\rightarrow
(16,\ 2)
$$

The batch dimension stays:

$$
16
$$

through all linear layers.

Only the feature dimension changes.


# 36. Using `nn.Sequential`

For simple feed-forward networks, we can also use:

`nn.Sequential`


In [ ]:
sequential_model = nn.Sequential(
    nn.Linear(3, 4),
    nn.ReLU(),
    nn.Linear(4, 2)
)

print(sequential_model)


In [ ]:
X = torch.randn(10, 3)

output = sequential_model(X)

print("Input shape:", X.shape)
print("Output shape:", output.shape)


# 37. `nn.Module` vs `nn.Sequential`

$$
\begin{array}{|c|c|}
\hline
\textbf{nn.Module} & \textbf{nn.Sequential} \\
\hline
\text{More flexible} & \text{Very concise} \\
\hline
\text{Custom forward logic} & \text{Simple ordered pipelines} \\
\hline
\text{Good for complex models} & \text{Good for simple stacks} \\
\hline
\end{array}
$$

Both are important.

We will use `nn.Module` often because it makes the forward logic explicit.


# 38. Raw Tensors vs `nn.Linear`

Earlier, we manually computed:

$$
XW^T+b
$$

`nn.Linear` performs the same basic operation.

Let's verify this.


In [ ]:
torch.manual_seed(0)

linear = nn.Linear(3, 2)

X = torch.randn(5, 3)

manual = X @ linear.weight.T + linear.bias
automatic = linear(X)

print("Manual shape:", manual.shape)
print("nn.Linear shape:", automatic.shape)
print("Outputs match:", torch.allclose(manual, automatic))


This is an important idea:

> PyTorch layers are not magic. They package tensor operations and learnable parameters into convenient modules.


# 39. Activation Functions Change Model Behavior

Consider a hidden pre-activation:

$$
z=
\begin{array}{|c|c|c|c|}
\hline
-2 & -0.5 & 1 & 3 \\
\hline
\end{array}
$$

After ReLU:

$$
\begin{array}{|c|c|c|c|}
\hline
0 & 0 & 1 & 3 \\
\hline
\end{array}
$$

The shape is unchanged, but the values are transformed nonlinearly.

This nonlinearity is what makes multi-layer networks powerful.


In [ ]:
z = torch.tensor([-2.0, -0.5, 1.0, 3.0])

print("Before ReLU:", z)
print("After ReLU:", torch.relu(z))


# 40. Activation Functions and Output Layers

A key beginner distinction:

> Hidden-layer activations and output-layer activations serve different purposes.

Examples:

## Regression

Often the final layer is linear:

$$
\text{output}\in(-\infty,\infty)
$$

## Binary Classification

Often one raw output logit is produced.

During training, `BCEWithLogitsLoss` is usually preferred.

## Multi-Class Classification

Usually the model produces one logit per class.

For 10 classes:

$$
output.shape=(batch,\ 10)
$$

During training, `CrossEntropyLoss` expects raw logits.

We will study loss functions carefully in a later notebook.


# 41. Example: 10-Class Classifier Shape

Suppose an image has already been converted into 784 features.

Input batch:

$$
X.shape=(32,\ 784)
$$

Model:

$$
784\rightarrow128\rightarrow10
$$

Then:

$$
(32,\ 784)
\rightarrow
(32,\ 128)
\rightarrow
(32,\ 10)
$$

The 10 output values correspond to 10 class scores.


In [ ]:
classifier = nn.Sequential(
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
)

X = torch.randn(32, 784)

logits = classifier(X)

print("Input shape:", X.shape)
print("Output logits shape:", logits.shape)


# 42. What Are Logits?

The raw outputs of a classification network are often called:

> **Logits**

For a 3-class model, one sample might produce:

$$
\begin{array}{|c|c|c|}
\hline
2.1 & -0.4 & 1.3 \\
\hline
\end{array}
$$

These are not necessarily probabilities.

Later, we will learn how losses such as:

`CrossEntropyLoss`

work directly with logits.


# 43. Common Beginner Mistakes

## Mistake 1 — Thinking a Neuron Is Only a Weighted Sum

A modern neuron usually includes:

- Linear combination
- Nonlinear activation

## Mistake 2 — Stacking Linear Layers Without Activations

Multiple purely linear layers collapse into one linear transformation.

## Mistake 3 — Using the Wrong Input Feature Size

If:

`nn.Linear(5, 10)`

then the last input dimension must be:

$$
5
$$

## Mistake 4 — Confusing Batch Size With Feature Size

For:

$$
(32,\ 128)
$$

usually:

- 32 = batch size
- 128 = features

## Mistake 5 — Applying ReLU to the Wrong Final Output

The correct output activation depends on the task.

## Mistake 6 — Forgetting That ReLU Preserves Shape

ReLU changes values, not dimensions.

## Mistake 7 — Calling `forward()` Directly

Usually use:

`model(x)`

instead of:

`model.forward(x)`

## Mistake 8 — Ignoring Parameter Shapes

Always inspect:

`model.named_parameters()`

when learning or debugging.


# 44. Neural-Network Debugging Checklist

When a network fails due to shape problems, inspect:

1. Input shape
2. Each `nn.Linear` input size
3. Each layer's weight shape
4. Output shape after every layer
5. Batch dimension
6. Feature dimension
7. Activation placement
8. Number of output units

Useful code:


In [ ]:
X = torch.randn(8, 3)

print("Input:", X.shape)

x1 = model.fc1(X)
print("After fc1:", x1.shape)

x2 = model.relu(x1)
print("After ReLU:", x2.shape)

x3 = model.fc2(x2)
print("After fc2:", x3.shape)


# 45. Practice Exercises

Try solving these before looking at the solutions.

## Exercise 1

A neuron receives:

$$
x=
\begin{array}{|c|c|}
\hline
2 & 3 \\
\hline
\end{array}
$$

Weights:

$$
w=
\begin{array}{|c|c|}
\hline
4 & -1 \\
\hline
\end{array}
$$

Bias:

$$
b=2
$$

Calculate the pre-activation value.

## Exercise 2

Apply ReLU to:

$$
\begin{array}{|c|c|c|c|}
\hline
-3 & -1 & 2 & 5 \\
\hline
\end{array}
$$

## Exercise 3

If:

$$
X.shape=(16,\ 5)
$$

and:

`nn.Linear(5, 8)`

what is the output shape?

## Exercise 4

For:

`nn.Linear(5,8)`

what are:

- Weight shape
- Bias shape
- Total number of parameters

## Exercise 5

Build:

$$
4\rightarrow6\rightarrow3
$$

using raw tensors.

## Exercise 6

Build the same network with `nn.Module`.

## Exercise 7

For input:

$$
(32,\ 4)
$$

predict the shapes through:

$$
4\rightarrow6\rightarrow3
$$

## Exercise 8

Build a classifier:

$$
784\rightarrow128\rightarrow10
$$

with ReLU in the hidden layer.

## Exercise 9

Count the total parameters in:

$$
784\rightarrow128\rightarrow10
$$

## Exercise 10

Explain why a network containing only linear layers cannot model general nonlinear relationships.


# 46. Shape Reasoning Challenges

Answer before running code.

## Challenge 1

Input:

$$
(64,\ 20)
$$

Layer:

`nn.Linear(20, 50)`

Output shape?

## Challenge 2

Input:

$$
(64,\ 50)
$$

Layer:

`nn.Linear(50, 10)`

Output shape?

## Challenge 3

Network:

$$
20\rightarrow50\rightarrow10
$$

Batch size:

$$
64
$$

Write the shape after every stage.

## Challenge 4

For:

`nn.Linear(100, 32)`

how many parameters are there including bias?

## Challenge 5

Network:

$$
5\rightarrow10\rightarrow10\rightarrow2
$$

How many weight matrices are there?

How many bias vectors?

## Challenge 6

If a model outputs:

$$
(32,\ 10)
$$

for a 10-class problem, what does each dimension represent?


# 47. Exercise Solutions


In [ ]:
# Exercise 1
x = torch.tensor([2.0, 3.0])
w = torch.tensor([4.0, -1.0])
b = torch.tensor(2.0)

z = torch.dot(x, w) + b
print("Exercise 1:", z)

# Exercise 2
x = torch.tensor([-3.0, -1.0, 2.0, 5.0])
print("Exercise 2:", torch.relu(x))

# Exercise 3
layer = nn.Linear(5, 8)
X = torch.randn(16, 5)
print("Exercise 3:", layer(X).shape)

# Exercise 4
print("Exercise 4 weight:", layer.weight.shape)
print("Exercise 4 bias:", layer.bias.shape)
print("Exercise 4 params:", sum(p.numel() for p in layer.parameters()))

# Exercise 5
X = torch.randn(7, 4)
W1 = torch.randn(6, 4)
b1 = torch.randn(6)
W2 = torch.randn(3, 6)
b2 = torch.randn(3)

h = torch.relu(X @ W1.T + b1)
out = h @ W2.T + b2
print("Exercise 5 output shape:", out.shape)

# Exercise 6
class ExerciseNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(4, 6)
        self.fc2 = nn.Linear(6, 3)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

exercise_model = ExerciseNetwork()
print("Exercise 6:", exercise_model)

# Exercise 7
X = torch.randn(32, 4)
h = exercise_model.fc1(X)
out = exercise_model.fc2(torch.relu(h))
print("Exercise 7 input:", X.shape)
print("Exercise 7 hidden:", h.shape)
print("Exercise 7 output:", out.shape)

# Exercise 8
classifier = nn.Sequential(
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
)
print("Exercise 8:", classifier)

# Exercise 9
print(
    "Exercise 9 parameters:",
    sum(p.numel() for p in classifier.parameters())
)


# 48. Key Takeaways

In this notebook, we learned:

- What a neuron is
- Inputs, weights, and bias
- Weighted sums
- Linear transformations
- Why purely linear networks are limited
- Why nonlinear activations matter
- ReLU
- Sigmoid
- Tanh
- Hidden layers
- Output layers
- Raw-tensor neural networks
- `nn.Module`
- `forward()`
- Batch processing
- `nn.Sequential`
- Parameter counting
- Shape reasoning through layers
- Logits
- Common neural-network mistakes

The central neural-network pattern is:

$$
\boxed{
\text{Linear}
\rightarrow
\text{Activation}
\rightarrow
\text{Linear}
}
$$

For deeper networks, we repeat:

$$
\boxed{
\text{Linear}
\rightarrow
\text{Activation}
}
$$

many times.


# 49. Check Your Understanding

Before moving forward, make sure you can answer these without searching:

1. What does a neuron compute?
2. What is the difference between $z$ and an activated output?
3. What do weights represent?
4. What does the bias do?
5. What is a linear transformation?
6. Why are multiple linear layers without activations still limited?
7. Why do neural networks need nonlinear activation functions?
8. What does ReLU do?
9. What range does sigmoid produce?
10. What range does tanh produce?
11. What is a hidden layer?
12. What is an output layer?
13. What does `nn.Linear(3,4)` mean?
14. What shape does `nn.Linear(3,4).weight` have?
15. What does `forward()` define?
16. Why should we usually call `model(x)` instead of `model.forward(x)`?
17. How do you count parameters in a linear layer?
18. Does ReLU change tensor shape?
19. What are logits?
20. Why is shape reasoning critical when building neural networks?


# Next Notebook

# 09 — `nn.Module` in Depth

In the next notebook, we will study:

- The `nn.Module` base class
- `__init__()`
- `forward()`
- Parameter registration
- `model.parameters()`
- `model.named_parameters()`
- `state_dict()`
- Submodules
- `nn.Sequential`
- Custom layers
- Training mode vs evaluation mode
- Saving and loading model weights
- Building reusable PyTorch model classes
